# RNN

This notebook trains an English-to-Igbo sequence-to-sequence model using a plain LSTM encoder-decoder.

It is written for local GPU training in VS Code. The default settings are tuned for faster local iteration rather than maximum model size.

## 1. Local setup

Expected local project layout:

- `CS156 Final Assignment/Cleaned Data/Combined_train.jsonl`
- `CS156 Final Assignment/Cleaned Data/Combined_test.jsonl`
- `CS156 Final Assignment/RNN Outputs/`

The notebook saves:

- `rnn_latest.pt`
- `rnn_best.pt`
- `rnn_final.pt`
- `rnn_history.csv`
- `rnn_test_metrics.csv`
- `rnn_test_preview.csv`
- `rnn_test_predictions.csv`

If training stops early, set `RESUME_TRAINING = True` and rerun from the top.

In [ ]:
# Run once in the active VS Code environment if needed.
#%pip install -q sacrebleu tqdm pandas

## 2. Imports and configuration

In [ ]:
from collections import Counter
from contextlib import nullcontext
from pathlib import Path
import json
import random
import re
import sys

import pandas as pd
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from IPython.display import display

try:
    from sacrebleu.metrics import BLEU, CHRF
except ModuleNotFoundError:
    BLEU = None
    CHRF = None
    print('sacrebleu is not installed. BLEU and chrF will be skipped until it is installed.')

IN_COLAB = 'google.colab' in sys.modules
MOUNT_DRIVE = True
SAVE_OUTPUTS_TO_DRIVE = True
DRIVE_PROJECT_SUBDIR = 'CS156 Final Assignment'
DRIVE_OUTPUT_SUBDIR = 'RNN Outputs'

if IN_COLAB and MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive') if IN_COLAB and MOUNT_DRIVE else None
DRIVE_PROJECT_DIR = DRIVE_ROOT / DRIVE_PROJECT_SUBDIR if DRIVE_ROOT is not None else None

PROJECT_DIR_CANDIDATES = []
if DRIVE_PROJECT_DIR is not None:
    PROJECT_DIR_CANDIDATES.append(DRIVE_PROJECT_DIR)

PROJECT_DIR_CANDIDATES.extend([
    Path.cwd(),
    Path.cwd().parent,
    Path(r'C:\Users\Mr. Paul\Downloads\CS156 Final Assignment'),
])


def resolve_project_dir(candidates):
    for candidate in candidates:
        if (candidate / 'Cleaned Data' / 'Combined_train.jsonl').exists():
            return candidate
    return None


PROJECT_DIR = resolve_project_dir(PROJECT_DIR_CANDIDATES)

if PROJECT_DIR is None:
    raise FileNotFoundError(
        'Could not locate the project directory. Set PROJECT_DIR manually before continuing.'
    )

DATA_DIR = PROJECT_DIR / 'Cleaned Data'
if IN_COLAB and SAVE_OUTPUTS_TO_DRIVE:
    if DRIVE_PROJECT_DIR is None:
        raise FileNotFoundError('Google Drive is not mounted, so outputs cannot be saved there.')
    OUTPUT_DIR = DRIVE_PROJECT_DIR / DRIVE_OUTPUT_SUBDIR
else:
    OUTPUT_DIR = PROJECT_DIR / 'RNN Outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / 'Combined_train.jsonl'
TEST_PATH = DATA_DIR / 'Combined_test.jsonl'

LATEST_CHECKPOINT_PATH = OUTPUT_DIR / 'rnn_latest.pt'
BEST_CHECKPOINT_PATH = OUTPUT_DIR / 'rnn_best.pt'
FINAL_CHECKPOINT_PATH = OUTPUT_DIR / 'rnn_final.pt'
HISTORY_PATH = OUTPUT_DIR / 'rnn_history.csv'
METRICS_PATH = OUTPUT_DIR / 'rnn_test_metrics.csv'
PREVIEW_PATH = OUTPUT_DIR / 'rnn_test_preview.csv'
PREDICTIONS_PATH = OUTPUT_DIR / 'rnn_test_predictions.csv'

RANDOM_SEED = 42

# All-row comparison configuration for local CUDA training.
# Long examples are truncated for the model instead of being dropped, so every pair is still used.
TRAIN_SAMPLE_SIZE = None
VALID_FRACTION = 0.02
# Smaller batches are slower, but materially safer on a 6 GB laptop GPU.
BATCH_SIZE = 64
EMBED_DIM = 256
HIDDEN_DIM = 384
NUM_LAYERS = 2
DROPOUT = 0.3
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 1e-5
NUM_EPOCHS = 10
MAX_SOURCE_TOKENS = 40
MAX_TARGET_TOKENS = 40
LENGTH_POLICY = 'truncate'

CLIP = 1.0
TEACHER_FORCING_RATIO = 0.6
MIN_FREQ = 3
MAX_DECODING_STEPS = 40
# Mixed precision and cuDNN RNN kernels are fast, but they have been unstable on this Windows 3060 setup.
FORCE_FP32_TRAINING = True
DISABLE_CUDNN_RNN = True

# Windows notebooks often run more reliably with a single-process DataLoader.
NUM_WORKERS = 0
REQUIRE_CUDA = True
RESUME_TRAINING = True
# Leave as None to evaluate the entire test set after training.
FULL_TEST_TRANSLATION_LIMIT = None

SPECIAL_TOKENS = {
    'pad': '<pad>',
    'sos': '<sos>',
    'eos': '<eos>',
    'unk': '<unk>',
}

random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = device.type == 'cuda' and not FORCE_FP32_TRAINING
# Windows Jupyter runs tend to be more stable without pinned host-memory transfers.
PIN_MEMORY = False

if REQUIRE_CUDA and device.type != 'cuda':
    raise RuntimeError(
        'CUDA is not available in the active kernel. Select the CUDA-enabled local Python environment before training.'
    )

if device.type == 'cuda':
    torch.set_float32_matmul_precision('high')
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    if DISABLE_CUDNN_RNN:
        torch.backends.cudnn.enabled = False

print('Running in Colab:', IN_COLAB)
print('Project directory:', PROJECT_DIR)
print('Data directory:', DATA_DIR)
print('Output directory:', OUTPUT_DIR)
print('Save outputs to Drive:', IN_COLAB and SAVE_OUTPUTS_TO_DRIVE)
print('Device:', device)
print('Training samples:', 'all rows' if TRAIN_SAMPLE_SIZE is None else TRAIN_SAMPLE_SIZE)
print('Epochs:', NUM_EPOCHS)
print('Length policy:', LENGTH_POLICY, f'({MAX_SOURCE_TOKENS}/{MAX_TARGET_TOKENS} token caps)')
print('Pin memory:', PIN_MEMORY)
print('Resume training:', RESUME_TRAINING)
print('AMP enabled:', USE_AMP)
print('cuDNN enabled:', torch.backends.cudnn.enabled)

if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
    print('CUDA build:', torch.version.cuda)
    print('GPU memory (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

## 3. File check

In [ ]:
print(f'Train file exists: {TRAIN_PATH.exists()} -> {TRAIN_PATH}')
print(f'Test file exists:  {TEST_PATH.exists()} -> {TEST_PATH}')

if not TRAIN_PATH.exists() or not TEST_PATH.exists():
    raise FileNotFoundError('The cleaned training or test file was not found.')

## 4. Data utilities

In [ ]:
def normalize_text(text):
    text = text.lower().strip()
    text = text.replace('\u2019', "'")
    text = text.replace('\u2018', "'")
    text = text.replace('\u201c', '"')
    text = text.replace('\u201d', '"')
    text = text.replace('\u02bc', "'")
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def tokenize(text):
    text = normalize_text(text)
    return re.findall(r"\w+|[^\w\s]", text, flags=re.UNICODE)


def read_jsonl(path):
    with path.open('r', encoding='utf-8') as handle:
        return [json.loads(line) for line in handle if line.strip()]


def get_source_tokens(row):
    return row.get('English_tokens', tokenize(row['English']))


def get_target_tokens(row):
    return row.get('Igbo_tokens', tokenize(row['Igbo']))


def prepare_examples(examples, max_source_tokens=None, max_target_tokens=None, length_policy='truncate'):
    prepared = []
    for example in examples:
        source_tokens = tokenize(example['English'])
        target_tokens = tokenize(example['Igbo'])

        if not source_tokens or not target_tokens:
            continue

        if length_policy == 'filter':
            if max_source_tokens is not None and len(source_tokens) > max_source_tokens:
                continue
            if max_target_tokens is not None and len(target_tokens) > max_target_tokens:
                continue
        elif length_policy == 'truncate':
            if max_source_tokens is not None:
                source_tokens = source_tokens[:max_source_tokens]
            if max_target_tokens is not None:
                target_tokens = target_tokens[:max_target_tokens]
        else:
            raise ValueError(f'Unsupported length policy: {length_policy}')

        prepared.append(
            {
                'English': normalize_text(example['English']),
                'Igbo': normalize_text(example['Igbo']),
                'English_tokens': source_tokens,
                'Igbo_tokens': target_tokens,
            }
        )
    return prepared


def split_train_validation(examples, val_fraction=0.02, seed=42):
    shuffled_examples = examples.copy()
    random.Random(seed).shuffle(shuffled_examples)
    val_size = max(1000, int(len(shuffled_examples) * val_fraction))
    val_examples = shuffled_examples[:val_size]
    train_examples = shuffled_examples[val_size:]
    return train_examples, val_examples

## 5. Load and split the data

In [ ]:
raw_train_examples = read_jsonl(TRAIN_PATH)
raw_test_examples = read_jsonl(TEST_PATH)

prepared_train_examples = prepare_examples(
    raw_train_examples,
    MAX_SOURCE_TOKENS,
    MAX_TARGET_TOKENS,
    LENGTH_POLICY,
)
prepared_test_examples = prepare_examples(
    raw_test_examples,
    MAX_SOURCE_TOKENS,
    MAX_TARGET_TOKENS,
    LENGTH_POLICY,
)

if TRAIN_SAMPLE_SIZE is not None:
    prepared_train_examples = random.Random(RANDOM_SEED).sample(prepared_train_examples, TRAIN_SAMPLE_SIZE)

train_examples, val_examples = split_train_validation(prepared_train_examples, VALID_FRACTION, RANDOM_SEED)
test_examples = prepared_test_examples

summary_df = pd.DataFrame(
    {
        'split': ['train', 'validation', 'test'],
        'rows': [len(train_examples), len(val_examples), len(test_examples)],
    }
)

display(summary_df)
display(pd.DataFrame(train_examples[:5]))

In [ ]:
length_df = pd.DataFrame(
    {
        'split': ['train', 'validation', 'test'],
        'avg_english_tokens': [
            round(sum(len(get_source_tokens(row)) for row in train_examples) / len(train_examples), 2),
            round(sum(len(get_source_tokens(row)) for row in val_examples) / len(val_examples), 2),
            round(sum(len(get_source_tokens(row)) for row in test_examples) / len(test_examples), 2),
        ],
        'avg_igbo_tokens': [
            round(sum(len(get_target_tokens(row)) for row in train_examples) / len(train_examples), 2),
            round(sum(len(get_target_tokens(row)) for row in val_examples) / len(val_examples), 2),
            round(sum(len(get_target_tokens(row)) for row in test_examples) / len(test_examples), 2),
        ],
    }
)

display(length_df)

## 6. Build vocabularies

In [ ]:
class Vocabulary:
    def __init__(self, min_freq=1):
        self.min_freq = min_freq
        self.itos = [
            SPECIAL_TOKENS['pad'],
            SPECIAL_TOKENS['sos'],
            SPECIAL_TOKENS['eos'],
            SPECIAL_TOKENS['unk'],
        ]
        self.stoi = {token: idx for idx, token in enumerate(self.itos)}

    def build(self, token_lists):
        counter = Counter(token for tokens in token_lists for token in tokens)
        for token, freq in sorted(counter.items()):
            if freq >= self.min_freq and token not in self.stoi:
                self.stoi[token] = len(self.itos)
                self.itos.append(token)

    def encode(self, tokens, add_special_tokens=True):
        ids = [self.stoi.get(token, self.unk_idx) for token in tokens]
        if add_special_tokens:
            ids = [self.sos_idx] + ids + [self.eos_idx]
        return ids

    def decode(self, ids):
        tokens = []
        for idx in ids:
            token = self.itos[idx]
            if token == SPECIAL_TOKENS['eos']:
                break
            if token not in SPECIAL_TOKENS.values():
                tokens.append(token)
        return ' '.join(tokens)

    @property
    def pad_idx(self):
        return self.stoi[SPECIAL_TOKENS['pad']]

    @property
    def sos_idx(self):
        return self.stoi[SPECIAL_TOKENS['sos']]

    @property
    def eos_idx(self):
        return self.stoi[SPECIAL_TOKENS['eos']]

    @property
    def unk_idx(self):
        return self.stoi[SPECIAL_TOKENS['unk']]

    def __len__(self):
        return len(self.itos)


train_source_tokens = [get_source_tokens(row) for row in train_examples]
train_target_tokens = [get_target_tokens(row) for row in train_examples]

source_vocab = Vocabulary(min_freq=MIN_FREQ)
target_vocab = Vocabulary(min_freq=MIN_FREQ)

source_vocab.build(train_source_tokens)
target_vocab.build(train_target_tokens)

print('Source vocabulary size:', len(source_vocab))
print('Target vocabulary size:', len(target_vocab))

## 7. Prepare tensor datasets and loaders

In [ ]:
class TranslationTensorDataset(Dataset):
    def __init__(self, examples, source_vocab, target_vocab):
        self.source_tensors = []
        self.target_tensors = []

        for row in examples:
            source_ids = source_vocab.encode(get_source_tokens(row))
            target_ids = target_vocab.encode(get_target_tokens(row))
            self.source_tensors.append(torch.tensor(source_ids, dtype=torch.long))
            self.target_tensors.append(torch.tensor(target_ids, dtype=torch.long))

    def __len__(self):
        return len(self.source_tensors)

    def __getitem__(self, idx):
        return self.source_tensors[idx], self.target_tensors[idx]


def collate_batch(batch):
    source_batch, target_batch = zip(*batch)
    source_lengths = torch.tensor([len(x) for x in source_batch], dtype=torch.long)

    sorted_indices = torch.argsort(source_lengths, descending=True)
    source_batch = [source_batch[i] for i in sorted_indices]
    target_batch = [target_batch[i] for i in sorted_indices]
    source_lengths = source_lengths[sorted_indices]

    source_batch = pad_sequence(source_batch, batch_first=True, padding_value=source_vocab.pad_idx)
    target_batch = pad_sequence(target_batch, batch_first=True, padding_value=target_vocab.pad_idx)

    return source_batch, source_lengths, target_batch


train_dataset = TranslationTensorDataset(train_examples, source_vocab, target_vocab)
val_dataset = TranslationTensorDataset(val_examples, source_vocab, target_vocab)
test_dataset = TranslationTensorDataset(test_examples, source_vocab, target_vocab)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_batch,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_batch,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_batch,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

print('Train batches:', len(train_loader))
print('Validation batches:', len(val_loader))
print('Test batches:', len(test_loader))

## 8. Define the plain LSTM encoder-decoder

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_dim, embed_dim, hidden_dim, num_layers, dropout, pad_idx):
        super().__init__()
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim

        self.embedding = nn.Embedding(input_dim, embed_dim, padding_idx=pad_idx)
        self.dropout = nn.Dropout(dropout)
        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=True,
            batch_first=True,
        )
        self.hidden_bridge = nn.Linear(hidden_dim * 2, hidden_dim)
        self.cell_bridge = nn.Linear(hidden_dim * 2, hidden_dim)

    def _combine_directions(self, state):
        combined_states = []
        for layer in range(self.num_layers):
            forward_state = state[2 * layer]
            backward_state = state[2 * layer + 1]
            combined_states.append(torch.cat([forward_state, backward_state], dim=1))
        return torch.stack(combined_states, dim=0)

    def forward(self, source, source_lengths):
        embedded = self.dropout(self.embedding(source))
        packed = pack_padded_sequence(embedded, source_lengths.cpu(), batch_first=True, enforce_sorted=True)
        _, (hidden, cell) = self.lstm(packed)

        hidden = torch.tanh(self.hidden_bridge(self._combine_directions(hidden)))
        cell = torch.tanh(self.cell_bridge(self._combine_directions(cell)))

        return hidden, cell


class Decoder(nn.Module):
    def __init__(self, output_dim, embed_dim, hidden_dim, num_layers, dropout, pad_idx):
        super().__init__()
        self.output_dim = output_dim
        self.embedding = nn.Embedding(output_dim, embed_dim, padding_idx=pad_idx)
        self.dropout = nn.Dropout(dropout)
        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.fc_out = nn.Linear(hidden_dim, output_dim)

    def forward(self, input_token, hidden, cell):
        input_token = input_token.unsqueeze(1)
        embedded = self.dropout(self.embedding(input_token))
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        prediction = self.fc_out(output.squeeze(1))
        return prediction, hidden, cell


class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, source, source_lengths, target, teacher_forcing_ratio=0.5):
        batch_size = source.size(0)
        target_length = target.size(1)
        target_vocab_size = self.decoder.output_dim

        outputs = torch.zeros(batch_size, target_length, target_vocab_size, device=self.device)
        hidden, cell = self.encoder(source, source_lengths)
        input_token = target[:, 0]

        for step in range(1, target_length):
            output, hidden, cell = self.decoder(input_token, hidden, cell)
            outputs[:, step] = output
            teacher_force = random.random() < teacher_forcing_ratio
            top_prediction = output.argmax(dim=1)
            input_token = target[:, step] if teacher_force else top_prediction

        return outputs

    def greedy_decode(self, source, source_lengths, sos_idx, eos_idx, max_steps):
        self.eval()
        with torch.no_grad():
            hidden, cell = self.encoder(source, source_lengths)
            input_token = torch.tensor([sos_idx], dtype=torch.long, device=self.device)
            generated_ids = []

            for _ in range(max_steps):
                output, hidden, cell = self.decoder(input_token, hidden, cell)
                next_token = output.argmax(dim=1)
                token_id = next_token.item()
                if token_id == eos_idx:
                    break
                generated_ids.append(token_id)
                input_token = next_token

        return generated_ids

## 9. Initialize the model and checkpoint helpers

In [ ]:
def amp_context():
    if USE_AMP:
        return torch.autocast(device_type='cuda', dtype=torch.float16)
    return nullcontext()


def count_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)


encoder = Encoder(
    input_dim=len(source_vocab),
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    pad_idx=source_vocab.pad_idx,
)

decoder = Decoder(
    output_dim=len(target_vocab),
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    pad_idx=target_vocab.pad_idx,
)

model = Seq2Seq(encoder, decoder, device).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=2,
)
criterion = nn.CrossEntropyLoss(ignore_index=target_vocab.pad_idx)
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

history = []
best_val_loss = float('inf')
start_epoch = 1


def save_checkpoint(path, epoch, best_val_loss, history):
    checkpoint = {
            'epoch': epoch,
            'best_val_loss': best_val_loss,
            'history': history,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'source_vocab_itos': source_vocab.itos,
            'target_vocab_itos': target_vocab.itos,
            'config': {
                'batch_size': BATCH_SIZE,
                'embed_dim': EMBED_DIM,
                'hidden_dim': HIDDEN_DIM,
                'num_layers': NUM_LAYERS,
                'dropout': DROPOUT,
                'teacher_forcing_ratio': TEACHER_FORCING_RATIO,
                'min_freq': MIN_FREQ,
                'max_source_tokens': MAX_SOURCE_TOKENS,
                'max_target_tokens': MAX_TARGET_TOKENS,
                'train_sample_size': TRAIN_SAMPLE_SIZE,
                'num_epochs': NUM_EPOCHS,
                'length_policy': LENGTH_POLICY,
            },
        }
    temp_path = path.with_suffix(path.suffix + '.tmp')
    torch.save(checkpoint, temp_path)
    temp_path.replace(path)


def save_history(history):
    history_df = pd.DataFrame(history)
    temp_path = HISTORY_PATH.with_suffix(HISTORY_PATH.suffix + '.tmp')
    history_df.to_csv(temp_path, index=False)
    temp_path.replace(HISTORY_PATH)


def load_resume_checkpoint():
    for candidate_path in (LATEST_CHECKPOINT_PATH, BEST_CHECKPOINT_PATH):
        if not candidate_path.exists():
            continue

        try:
            checkpoint = torch.load(candidate_path, map_location=device)
            print(f"Loaded checkpoint from {candidate_path.name} (epoch {checkpoint['epoch']}).")
            return checkpoint
        except Exception as exc:
            print(f'Warning: could not load {candidate_path.name}: {exc}')

    return None


if RESUME_TRAINING:
    checkpoint = load_resume_checkpoint()
    if checkpoint is not None:
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        if USE_AMP:
            scaler.load_state_dict(checkpoint['scaler_state_dict'])
        else:
            print('AMP is disabled for this run; ignoring the saved GradScaler state.')
        history = checkpoint['history']
        best_val_loss = checkpoint['best_val_loss']
        start_epoch = checkpoint['epoch'] + 1
        print(f'Resuming training from epoch {start_epoch}.')

print('Trainable parameters:', f'{count_parameters(model):,}')

## 10. Training and evaluation helpers

In [ ]:
def train_epoch(model, dataloader, optimizer, criterion, clip, scaler, teacher_forcing_ratio):
    model.train()
    epoch_loss = 0.0

    for source_batch, source_lengths, target_batch in tqdm(dataloader, desc='Training', leave=False):
        source_batch = source_batch.to(device, non_blocking=PIN_MEMORY)
        target_batch = target_batch.to(device, non_blocking=PIN_MEMORY)

        optimizer.zero_grad(set_to_none=True)

        with amp_context():
            output = model(source_batch, source_lengths, target_batch, teacher_forcing_ratio=teacher_forcing_ratio)
            output_dim = output.size(-1)
            output = output[:, 1:].reshape(-1, output_dim)
            target = target_batch[:, 1:].reshape(-1)
            loss = criterion(output, target)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item()
        del output, target, loss

    if device.type == 'cuda':
        torch.cuda.empty_cache()

    return epoch_loss / len(dataloader)


def evaluate_loss(model, dataloader, criterion):
    model.eval()
    epoch_loss = 0.0

    with torch.no_grad():
        for source_batch, source_lengths, target_batch in tqdm(dataloader, desc='Evaluating loss', leave=False):
            source_batch = source_batch.to(device, non_blocking=PIN_MEMORY)
            target_batch = target_batch.to(device, non_blocking=PIN_MEMORY)

            with amp_context():
                output = model(source_batch, source_lengths, target_batch, teacher_forcing_ratio=0.0)
                output_dim = output.size(-1)
                output = output[:, 1:].reshape(-1, output_dim)
                target = target_batch[:, 1:].reshape(-1)
                loss = criterion(output, target)

            epoch_loss += loss.item()
            del output, target, loss

    if device.type == 'cuda':
        torch.cuda.empty_cache()

    return epoch_loss / len(dataloader)


def translate_text(model, text, source_vocab, target_vocab, max_steps=MAX_DECODING_STEPS):
    tokens = tokenize(text)
    if MAX_SOURCE_TOKENS is not None:
        tokens = tokens[:MAX_SOURCE_TOKENS]
    source_ids = source_vocab.encode(tokens)
    source_tensor = torch.tensor(source_ids, dtype=torch.long, device=device).unsqueeze(0)
    source_length = torch.tensor([len(source_ids)], dtype=torch.long)
    generated_ids = model.greedy_decode(source_tensor, source_length, target_vocab.sos_idx, target_vocab.eos_idx, max_steps)
    return target_vocab.decode(generated_ids)


def evaluate_translations(model, examples, source_vocab, target_vocab, sample_limit=None):
    rows = examples[:sample_limit] if sample_limit is not None else examples
    predictions = []
    references = []
    records = []

    for item in tqdm(rows, desc='Generating translations'):
        predicted_text = translate_text(model, item['English'], source_vocab, target_vocab)
        reference_text = ' '.join(get_target_tokens(item))

        predictions.append(predicted_text)
        references.append([reference_text])
        records.append(
            {
                'English': item['English'],
                'Reference Igbo': reference_text,
                'Predicted Igbo': predicted_text,
            }
        )

    exact_match = sum(pred == refs[0] for pred, refs in zip(predictions, references)) / len(predictions)
    bleu_score = BLEU().corpus_score(predictions, references).score if BLEU is not None else float('nan')
    chrf_score = CHRF().corpus_score(predictions, references).score if CHRF is not None else float('nan')

    metrics_df = pd.DataFrame(
        {
            'metric': ['Exact match', 'BLEU', 'chrF'],
            'value': [round(exact_match, 4), round(bleu_score, 2), round(chrf_score, 2)],
        }
    )

    predictions_df = pd.DataFrame(records)
    preview_df = predictions_df.head(20)
    return metrics_df, predictions_df, preview_df

## 11. Train the model

This loop uses validation loss, not test loss, for checkpoint selection.

In [ ]:
if start_epoch > NUM_EPOCHS:
    print(f'Checkpoint already reached epoch {start_epoch - 1}; skipping the training loop.')
else:
    for epoch in range(start_epoch, NUM_EPOCHS + 1):
        train_loss = train_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
            CLIP,
            scaler,
            TEACHER_FORCING_RATIO,
        )
        val_loss = evaluate_loss(model, val_loader, criterion)
        scheduler.step(val_loss)

        learning_rate = optimizer.param_groups[0]['lr']
        history.append(
            {
                'epoch': epoch,
                'train_loss': train_loss,
                'val_loss': val_loss,
                'learning_rate': learning_rate,
            }
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            save_checkpoint(BEST_CHECKPOINT_PATH, epoch, best_val_loss, history)

        save_checkpoint(LATEST_CHECKPOINT_PATH, epoch, best_val_loss, history)
        save_history(history)

        print(
            f'Epoch {epoch:02d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | lr={learning_rate:.6f}'
        )

history_df = pd.DataFrame(history)
display(history_df)

## 12. Load the best checkpoint and evaluate on the test set

In [ ]:
best_checkpoint = torch.load(BEST_CHECKPOINT_PATH, map_location=device)
model.load_state_dict(best_checkpoint['model_state_dict'])

test_loss = evaluate_loss(model, test_loader, criterion)
metrics_df, predictions_df, preview_df = evaluate_translations(
    model,
    test_examples,
    source_vocab,
    target_vocab,
    sample_limit=FULL_TEST_TRANSLATION_LIMIT,
)

metrics_df.loc[len(metrics_df)] = ['Test loss', round(test_loss, 4)]

display(metrics_df)
display(preview_df)

metrics_df.to_csv(METRICS_PATH, index=False)
preview_df.to_csv(PREVIEW_PATH, index=False)
predictions_df.to_csv(PREDICTIONS_PATH, index=False)

## 13. Save the final checkpoint

In [ ]:
save_checkpoint(FINAL_CHECKPOINT_PATH, history[-1]['epoch'], best_val_loss, history)

saved_files = pd.DataFrame(
    {
        'artifact': [
            'latest checkpoint',
            'best checkpoint',
            'final checkpoint',
            'training history',
            'test metrics',
            'preview predictions',
            'test predictions',
        ],
        'path': [
            str(LATEST_CHECKPOINT_PATH),
            str(BEST_CHECKPOINT_PATH),
            str(FINAL_CHECKPOINT_PATH),
            str(HISTORY_PATH),
            str(METRICS_PATH),
            str(PREVIEW_PATH),
            str(PREDICTIONS_PATH),
        ],
    }
)

display(saved_files)

## 14. Simple inference examples

In [ ]:
sample_sentences = [
    'good morning',
    'god is good',
    'where are you going',
    'the lord is my shepherd',
]

for sentence in sample_sentences:
    prediction = translate_text(model, sentence, source_vocab, target_vocab)
    print(f'English: {sentence}')
    print(f'Predicted Igbo: {prediction}')
    print('-' * 60)